
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Using MLflow Built-In Judges
## Overview
In this demo, we'll use MLflow's built-in `Correctness` and `Safety` judges to evaluate the Airbnb agent. You'll create judge instances, load evaluation datasets, run `mlflow.genai.evaluate()`, and inspect the results.

## Learning Objectives
By the end of this demo, you will be able to:
1. Configure built-in judges for correctness and safety assessment
2. Execute evaluations using `mlflow.genai.evaluate()` with multiple scorers
3. Interpret evaluation results using MLflow's UI and result dataframes

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This demo uses the agent created in <strong>Demo - Agent Setup</strong>. Please ensure you have completed that notebook before proceeding.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to configure your environment.

In [0]:
%run ../Includes/Classroom-Setup-2

In [0]:
print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
print(f"Schema Name:       {schema_name}")

## B. Correctness Evaluation

The `Correctness` judge assesses whether the agent's response is factually correct by comparing it against ground truth defined as `expected_facts` or `expected_response`.

We'll specify a model endpoint for the judge using the `model` parameter with the format `databricks:/<endpoint-name>`.

In [0]:
from mlflow.genai.scorers import Correctness

correctness_eval = Correctness(
    model=correctness_eval_endpoint)

### B1. Load Evaluation Dataset

The correctness evaluation dataset is stored in the UC volume. Each entry contains `inputs` (the query) and `expectations` with `expected_facts` that the response should contain.

In [0]:
import json
from pprint import pprint
from pathlib import Path

path = Path(f"/Volumes/{catalog_name}/{schema_name}/agent_vol/correctness_eval.json")

with path.open("r", encoding="utf-8") as f:
    eval_dataset = json.load(f)

pprint(eval_dataset)

### B2. Run Correctness Evaluation

`mlflow.genai.evaluate()` takes three key parameters:
- **data**: The evaluation dataset
- **predict_fn**: A function that calls the agent for each input
- **scorers**: The list of judges to apply

After running the cell, click **View evaluation results in MLflow** to inspect the traces and feedback.

In [0]:
correctness_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval],
)

### B3. Inspect Results

The `EvaluationResult` object contains `metrics` (aggregated scores) and `result_df` (per-row details with values and rationales). Click on any row to view the full MLflow trace.

In [0]:
print(f"The run ID is: {correctness_results.run_id}")
print(f"The aggregated metrics are: {correctness_results.metrics}")
print("\nThe results from the previous batch of inputs:")
display(getattr(correctness_results, "result_df", None))

## C. Safety Evaluation

The `Safety` judge checks whether content is free from harmful, offensive, or toxic material. Unlike Correctness, it does not require ground truth: it evaluates the agent's response directly.

In [0]:
from mlflow.genai.scorers import Safety

safety_eval = Safety(
    model=safety_eval_endpoint)


### C1. Load Safety Evaluation Dataset

Unlike the correctness dataset, the safety dataset does not require `expected_facts` or ground truth. The judge evaluates the response in isolation, checking for harmful, offensive, or toxic content regardless of factual accuracy.

In [0]:
path = Path(f"/Volumes/{catalog_name}/{schema_name}/agent_vol/safety_eval.json")

with path.open("r", encoding="utf-8") as f:
    safety_eval_dataset = json.load(f)

pprint(safety_eval_dataset)


### C2. Run Safety Evaluation

The same `mlflow.genai.evaluate()` pattern applies here. Since `Safety` doesn't need ground truth, the dataset only needs `inputs`: making it easy to generate adversarial test cases that probe the agent's guardrails.

In [0]:
safety_results = mlflow.genai.evaluate(
    data=safety_eval_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[safety_eval],
)


### C3. Inspect Results

Safety scores are binary: `yes` (safe) or `no` (unsafe). The rationale explains which safety category was flagged (if any). Look for cases where the agent might have been tricked into producing harmful content despite its system prompt.

In [0]:
print(f"The run ID is: {safety_results.run_id}")
print(f"The aggregated metrics are: {safety_results.metrics}")
print("\nThe results from the safety evaluation:")
display(getattr(safety_results, "result_df", None))


## D. Evaluating Multiple Scorers at Once

You can pass multiple scorers in a single `mlflow.genai.evaluate()` call. All metrics are logged to the same evaluation run, which means:
- A single set of traces is generated (reducing LLM calls to the agent)
- All judge scores appear side-by-side in the results dataframe
- The MLflow UI displays a unified view for comparing across dimensions

This is the recommended pattern for production evaluation pipelines where you want to assess multiple quality dimensions simultaneously.

In [0]:
scorers = [safety_eval, correctness_eval]
mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=scorers
)

## E. Conclusion
In this demo, you:

1. Configured built-in `Correctness` and `Safety` judges with custom model endpoints
2. Loaded evaluation datasets from a UC volume and ran evaluations with `mlflow.genai.evaluate()`
3. Inspected per-row results including scores, rationales, and traces
4. Combined multiple scorers in a single evaluation run

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
